### Configuração

In [1]:
import pandas as pd
from gensim import corpora
import re
import os
from gensim import corpora, models

# Carrega o dataset
df_novo = pd.read_csv('./datasets/database-lemmetizado.csv.zip', compression='zip')
df_novo = df_novo.sort_values('date_published').reset_index(drop=True)

# Converter a coluna de datas para datetime
df_novo['date_published'] = pd.to_datetime(df_novo['date_published'], errors='coerce')

# Usar esse caso o dataset esteja pre-processado
def tokenize_text(text):
    """
    Função para tokenizar o texto.
    """
    # Converte para minúsculas e separa em palavras
    tokens = text.lower().split()
    return tokens

documents = df_novo['tokens'].dropna().astype(str).tolist()
processed_docs = [tokenize_text(doc) for doc in documents]

# Cria dicionário e corpus
dictionary = corpora.Dictionary(processed_docs)
corpus = [dictionary.doc2bow(doc) for doc in processed_docs]
    
print(f"Número de documentos: {len(corpus)}")
print(f"Tamanho do dicionário: {len(dictionary)}")
print(f"Exemplo de documento (Bag-of-Words): {corpus[0]}")
print(f"processed_docs[0]: {processed_docs[0]}")

Número de documentos: 25240
Tamanho do dicionário: 47851
Exemplo de documento (Bag-of-Words): [(0, 1), (1, 1), (2, 1), (3, 4), (4, 2), (5, 2), (6, 1), (7, 4), (8, 1), (9, 3), (10, 1), (11, 1), (12, 1), (13, 1), (14, 1), (15, 1), (16, 1), (17, 1), (18, 3), (19, 1), (20, 1), (21, 1), (22, 1), (23, 1), (24, 1), (25, 1), (26, 1), (27, 1), (28, 1), (29, 1), (30, 1), (31, 1), (32, 1), (33, 1), (34, 4), (35, 1), (36, 1)]
processed_docs[0]: ["['dispositivo',", "'eletrocrèmico',", "'combinação',", "'eletrocrèmica',", "'patente',", "'invenção',", "'dispositivo',", "'eletrocrèmico',", "'combinação',", "'eletrocrèmico',", "'apresentar',", "'dispositivo',", "'eletrocrómico',", "'aplicação',", "'superfície',", "'dispositivo',", "'utilizar',", "'camada',", "'regulação',", "'óptico',", "'modo',", "'reduzir',", "'pequeno',", "'interferência',", "'óptico',", "'camada',", "'estrutura',", "'modo',", "'aumentar',", "'transparência',", "'óptico',", "'uniforme',", "'sintonização',", "'óptico',", "'permitir',

### Treinamento LDA

In [15]:
def train_and_save_lda_per_year(df, num_topics=10, passes=10, alpha='auto', eta='auto'):
    """
    Treina um modelo LDA para cada ano e salva-o individualmente no disco.
    A função foca apenas na execução, sem retornar estatísticas.

    Args:
        df (pd.DataFrame): DataFrame com 'date_published' (datetime) e 'tokens'.
        num_topics (int): Número de tópicos para cada modelo LDA.
        passes (int): Número de passes de treinamento.
        alpha (str or float): Parâmetro alpha do LDA.
        eta (str or float): Parâmetro eta do LDA.
    """
    
    # Extrai o ano das datas
    df['year'] = df['date_published'].dt.year
    years = sorted(df['year'].dropna().unique())
    
    print(f"Anos encontrados no dataset: {years}")
    
    # Define o diretório onde os modelos serão salvos
    output_dir = f'./modelos/lda_por_ano/{num_topics}_topics/'
    os.makedirs(output_dir, exist_ok=True)
    print(f"Modelos serão salvos em: '{output_dir}'")
    
    models_trained_count = 0
    
    for year in years:
        print(f"\n=== Processando ano: {year} ===")
        
        df_year = df[df['year'] == year].copy()
        
        documents = df_year['tokens'].dropna().tolist()
        
        processed_docs = [doc.split() if isinstance(doc, str) else doc for doc in documents]
        
        processed_docs = [doc for doc in processed_docs if doc]
        
        print(f"Documentos para treinamento: {len(processed_docs)}")
        
        dictionary_year = corpora.Dictionary(processed_docs)
        corpus_year = [dictionary_year.doc2bow(doc) for doc in processed_docs]
        
        try:
            # Treina o modelo LDA
            lda_model = models.LdaModel(
                corpus=corpus_year,
                id2word=dictionary_year,
                num_topics=num_topics,
                random_state=42,
                passes=passes,
                alpha=alpha,
                eta=eta
            )
            
            # Salva o modelo treinado
            model_path = os.path.join(output_dir, f'lda_model_{year}.model')
            lda_model.save(model_path)
            
            print(f"Modelo para o ano {year} treinado e salvo com sucesso em '{model_path}'")
            models_trained_count += 1
            
        except Exception as e:
            print(f"ERRO ao treinar o modelo para o ano {year}: {e}")
            continue
    
    print("\n=== Treinamento Concluído ===")
    print(f"Total de modelos treinados e salvos: {models_trained_count}")

train = False
number_of_topics = 5
if train:
    print("Iniciando o processo de treinamento de modelos LDA por ano...")
    train_and_save_lda_per_year(
        df=df_novo,
        num_topics=number_of_topics,
        passes=10
    )

    print("\nProcesso finalizado.")

### Gráficos

In [16]:
from gensim.models import LdaModel

def display_topics_from_models_with_probs(model_dir, num_words=10):
    """
    Carrega modelos LDA de um diretório e exibe os tópicos de cada um,
    incluindo a probabilidade de cada palavra formatada como percentagem.

    Args:
        model_dir (str): O caminho para o diretório onde os modelos .model estão salvos.
        num_words (int): O número de palavras a serem exibidas para cada tópico.
    """
    
    if not os.path.exists(model_dir):
        print(f"ERRO: O diretório '{model_dir}' não foi encontrado.")
        return

    try:
        model_files = [f for f in os.listdir(model_dir) if f.startswith('lda_model_') and f.endswith('.model')]
    except FileNotFoundError:
        print(f"ERRO: O diretório '{model_dir}' não foi encontrado.")
        return

    if not model_files:
        print(f"Nenhum ficheiro de modelo (.model) encontrado em '{model_dir}'.")
        return
        
    model_files.sort()
    
    print(f"Encontrados {len(model_files)} modelos. Exibindo os tópicos para cada ano...")
    
    for filename in model_files:
        try:
            match = re.search(r'_(\d{4})\.model', filename)
            if not match:
                continue
            
            year = match.group(1)
            model_path = os.path.join(model_dir, filename)
            lda_model = LdaModel.load(model_path)
            
            print(f"\n--- Tópicos para o Ano: {year} ---")
            
            topics = lda_model.show_topics(num_topics=-1, num_words=num_words, formatted=False)
            
            for topic_id, word_probs in topics:
                # Formata cada palavra com a sua probabilidade em percentagem
                formatted_words = [f"{word} ({prob*100:.2f}%)" for word, prob in word_probs]
                
                # Junta as palavras já formatadas numa única string para exibição
                print(f"Tópico {topic_id}: {', '.join(formatted_words)}")
                
        except Exception as e:
            print(f"\nERRO ao processar o ficheiro {filename}: {e}")
            continue

WORDS_PER_TOPIC = 8
MODEL_DIRECTORY = f'./modelos/lda_por_ano/{number_of_topics}_topics/'

display_topics_from_models_with_probs(MODEL_DIRECTORY, num_words=WORDS_PER_TOPIC)

Encontrados 26 modelos. Exibindo os tópicos para cada ano...

--- Tópicos para o Ano: 1999 ---
Tópico 0: 'porta', (3.82%), 'dobradiça', (1.51%), 'motor', (1.09%), 'elevador', (1.01%), 'carro', (0.72%), 'acionamento', (0.66%), 'pino', (0.64%), 'abertura', (0.63%)
Tópico 1: 'invenção', (0.93%), 'elemento', (0.85%), 'material', (0.82%), 'fixação', (0.77%), 'conjunto', (0.73%), 'patente', (0.63%), 'fixador', (0.61%), 'painel', (0.57%)
Tópico 2: 'inferior', (1.07%), 'água', (0.99%), 'superfície', (0.76%), 'parede', (0.74%), 'extremidade', (0.67%), 'lateral', (0.63%), 'aba', (0.58%), 'possuir', (0.57%)
Tópico 3: 'elemento', (0.78%), 'invenção', (0.77%), 'dispositivo', (0.67%), 'parede', (0.66%), 'painel', (0.60%), 'tubo', (0.60%), 'fecho', (0.59%), 'l', (0.56%)
Tópico 4: 'construção', (1.23%), 'elemento', (1.07%), 'parede', (1.03%), 'invenção', (0.96%), 'estrutura', (0.69%), 'vedação', (0.58%), 'patente', (0.57%), 'material', (0.56%)

--- Tópicos para o Ano: 2000 ---
Tópico 0: 'porta', (0.85

### Processamento das patentes

In [17]:
import pandas as pd
from gensim.corpora import Dictionary
from gensim.models import LdaModel
import warnings
import json
from pandarallel import pandarallel

pandarallel.initialize(progress_bar=True)

warnings.filterwarnings("ignore", category=DeprecationWarning)

def _processar_patente(row, lda_model, dicionario, distribuicoes_topicos_ano, num_topics):
    """
    Função auxiliar para processar UMA ÚNICA patente (uma linha do DataFrame).
    """
    import pandas as pd
    import json
    lens_id_final = row['lens_id']
    
    tokens_brutos = row['tokens']
    if pd.isna(tokens_brutos) or tokens_brutos == '':
        return None
        
    if isinstance(tokens_brutos, str):
        tokens_patente = tokens_brutos.split()
    elif isinstance(tokens_brutos, list):
        tokens_patente = tokens_brutos
    else:
        return None
    
    # Filtra tokens vazios
    tokens_patente = [t for t in tokens_patente if t and isinstance(t, str)]
    
    if not tokens_patente:
        return None

    # Distribuição de tópicos para esta patente
    doc_bow = dicionario.doc2bow(tokens_patente)
    distribuicao_topicos_patente = lda_model.get_document_topics(doc_bow, minimum_probability=0.0)
    prob_topico_dado_doc = {topico_id: prob for topico_id, prob in distribuicao_topicos_patente}

    resultado_patente = {
        'lens_id': lens_id_final, 
        'year': int(row['year'])
    }
    
    # Calcular a pontuação para cada palavra em cada tópico
    for id_topico in range(num_topics):
        p_topico_na_patente = prob_topico_dado_doc.get(id_topico, 0)

        distribuicao_palavras_topico = distribuicoes_topicos_ano[id_topico]
        
        lista_pontuacao_palavras = []
        for id_palavra, p_palavra_no_topico in distribuicao_palavras_topico:
            # Verifica se a palavra está no documento
            if id_palavra in [word_id for word_id, _ in doc_bow]:
                pontuacao = float(p_topico_na_patente * p_palavra_no_topico)
                palavra_str = dicionario[id_palavra]
                lista_pontuacao_palavras.append((palavra_str, pontuacao))
        
        # Ordena e pega apenas as top palavras
        lista_pontuacao_palavras.sort(key=lambda item: item[1], reverse=True)
        top_palavras = lista_pontuacao_palavras[:50]
        
        resultado_patente[f'Topic_{id_topico}'] = json.dumps(top_palavras, ensure_ascii=False)
    return resultado_patente

INFO: Pandarallel will run on 4 workers.
INFO: Pandarallel will use standard multiprocessing data transfer (pipe) to transfer data between the main process and workers.

https://nalepae.github.io/pandarallel/troubleshooting/


In [18]:
def analisar_e_salvar_por_ano_aprimorado(df, modelos_dir, num_topics, output_dir, 
                                          anos_para_rodar=None, ignorar_existentes=True):
    """
    Processa as patentes ano por ano de forma otimizada.
    """
    import pandas as pd
    from gensim.models import LdaModel
    import os
    # A parte do salvar os resultado foi gerada pelo gemini, não modfifiquei
    os.makedirs(output_dir, exist_ok=True)
    
    colunas_necessarias = ['lens_id', 'year', 'tokens']
    for col in colunas_necessarias:
        if col not in df.columns:
            raise ValueError(f"Coluna '{col}' não encontrada no DataFrame!")
    
    if df['lens_id'].duplicated().any():
        print(f"AVISO: {df['lens_id'].duplicated().sum()} lens_id duplicados encontrados. Removendo...")
        df = df.drop_duplicates(subset=['lens_id'], keep='first')
    
    anos_alvo = []
    if anos_para_rodar:
        if isinstance(anos_para_rodar, int):
            anos_alvo = [anos_para_rodar]
        else:
            anos_alvo = sorted(anos_para_rodar)
        print(f"Análise direcionada para os anos específicos: {anos_alvo}")
    else:
        anos_alvo = sorted(df['year'].dropna().unique().astype(int))
        print(f"Iniciando análise para todos os anos encontrados: {anos_alvo}")

    for ano in anos_alvo:
        if ignorar_existentes:
            caminho_parquet = os.path.join(output_dir, f'resultados_{ano}.parquet')
            caminho_csv = os.path.join(output_dir, f'resultados_{ano}.csv')
            if os.path.exists(caminho_parquet) or os.path.exists(caminho_csv):
                print(f"\n--- Ano {ano} já processado. Pulando... ---")
                continue

        print(f"\n{'='*60}")
        print(f"Processando o ano: {ano}")
        print(f"{'='*60}")
        
        caminho_modelo = os.path.join(modelos_dir, f'lda_model_{ano}.model')
        
        if not os.path.exists(caminho_modelo):
            print(f"AVISO: Modelo para o ano {ano} não encontrado em {caminho_modelo}")
            continue
            
        try:
            lda_model_ano = LdaModel.load(caminho_modelo)
            dicionario_ano = lda_model_ano.id2word
            print(f"✓ Modelo carregado com sucesso")
        except Exception as e:
            print(f"✗ ERRO ao carregar o modelo: {e}")
            continue

        print("Pré-calculando distribuições de palavras por tópico...")
        distribuicoes_topicos_ano = {
            id_topico: lda_model_ano.get_topic_terms(id_topico, topn=500)
            for id_topico in range(num_topics)
        }
        
        df_ano = df[df['year'] == ano].copy()
        print(f"✓ {len(df_ano)} patentes encontradas para processar")

        if df_ano.empty:
            print("Nenhuma patente para processar. Pulando...")
            continue
        
        print("Processando patentes...")
        try:
            resultados_do_ano_series = df_ano.parallel_apply(
                _processar_patente, 
                axis=1, 
                lda_model=lda_model_ano, 
                dicionario=dicionario_ano,
                distribuicoes_topicos_ano=distribuicoes_topicos_ano, 
                num_topics=num_topics
            )
        except Exception as e:
            print(f"✗ ERRO durante o processamento: {e}")
            import traceback
            traceback.print_exc()
            continue
        
        resultados_do_ano_lista = [res for res in resultados_do_ano_series if res is not None]
        
        if not resultados_do_ano_lista:
            print("Nenhum resultado válido gerado. Pulando...")
            continue
        
        print(f"✓ {len(resultados_do_ano_lista)} patentes processadas com sucesso")
            
        df_resultados_ano = pd.DataFrame(resultados_do_ano_lista)
        
        caminho_saida_parquet = os.path.join(output_dir, f'resultados_{ano}.parquet')
        caminho_saida_csv = os.path.join(output_dir, f'resultados_{ano}.csv')
        
        try:
            df_resultados_ano.to_parquet(caminho_saida_parquet, index=False, engine='pyarrow')
            print(f"✓ Resultados salvos em Parquet: {caminho_saida_parquet}")
        except Exception as e:
            print(f"⚠ Não foi possível salvar em Parquet, usando CSV: {str(e)[:100]}")
            try:
                df_resultados_ano.to_csv(caminho_saida_csv, index=False)
                print(f"✓ Resultados salvos em CSV: {caminho_saida_csv}")
            except Exception as e2:
                print(f"✗ ERRO ao salvar resultados: {e2}")

    print("\n" + "="*60)
    print("PROCESSAMENTO CONCLUÍDO")
    print("="*60)

In [19]:
if 'year' not in df_novo.columns:
    df_novo['year'] = df_novo['date_published'].dt.year

if 'index' in df_novo.columns:
    df_novo = df_novo.drop(columns=['index'])

df_novo = df_novo.reset_index(drop=True)

print(f"Colunas do DataFrame: {df_novo.columns.tolist()}")
print(f"Número de patentes: {len(df_novo)}")

Colunas do DataFrame: ['lens_id', 'jurisdiction', 'doc_number', 'kind', 'date_published', 'doc_key', 'docdb_id', 'lang', 'biblio', 'families', 'legal_status', 'publication_type', 'abstract', 'abstract_text', 'ipcr_triples', 'cpc_triples', 'all_triples', 'inventors_detailed', 'inventor_names', 'invention_title_text', 'patent_count', 'patent_lens_ids', 'patent_status', 'application_expiry_date', 'picked', 'title_abstract', 'tokens', 'year']
Número de patentes: 25240


In [20]:
diretorio_dos_modelos = f'./modelos/lda_por_ano/{number_of_topics}_topics/'
diretorio_de_saida = f'./resultados_por_ano/{number_of_topics}_topics/'

print("\nIniciando o processo de análise e salvamento por ano...")

analisar_e_salvar_por_ano_aprimorado(
    df=df_novo,
    modelos_dir=diretorio_dos_modelos,
    num_topics=number_of_topics,
    output_dir=diretorio_de_saida
)

print("\nProcesso finalizado. Os resultados estão salvos na pasta 'resultados_por_ano'.")


Iniciando o processo de análise e salvamento por ano...
Iniciando análise para todos os anos encontrados: [1999, 2000, 2001, 2002, 2003, 2004, 2005, 2006, 2007, 2008, 2009, 2010, 2011, 2012, 2013, 2014, 2015, 2016, 2017, 2018, 2019, 2020, 2021, 2022, 2023, 2024]

--- Ano 1999 já processado. Pulando... ---

--- Ano 2000 já processado. Pulando... ---

--- Ano 2001 já processado. Pulando... ---

--- Ano 2002 já processado. Pulando... ---

--- Ano 2003 já processado. Pulando... ---

--- Ano 2004 já processado. Pulando... ---

--- Ano 2005 já processado. Pulando... ---

--- Ano 2006 já processado. Pulando... ---

--- Ano 2007 já processado. Pulando... ---

--- Ano 2008 já processado. Pulando... ---

--- Ano 2009 já processado. Pulando... ---

--- Ano 2010 já processado. Pulando... ---

--- Ano 2011 já processado. Pulando... ---

--- Ano 2012 já processado. Pulando... ---

--- Ano 2013 já processado. Pulando... ---

--- Ano 2014 já processado. Pulando... ---

--- Ano 2015 já processado. Pul

### Gerar dataframe com os resultados

In [21]:
from glob import glob

# Função para concatenar resultados de múltiplos arquivos
# Feito pelo gemini, não modifiquei
def concatenar_resultados_por_ano(diretorio_resultados, formato='csv'):
    """
    Concatena todos os arquivos de resultados (CSV ou Parquet) em um único DataFrame.
    
    Args:
        diretorio_resultados (str): Caminho do diretório com os arquivos
        formato (str): 'csv', 'parquet' ou 'auto' (tenta ambos)
    
    Returns:
        pd.DataFrame: DataFrame concatenado com todos os anos
    """
    
    if not os.path.exists(diretorio_resultados):
        raise ValueError(f"Diretório '{diretorio_resultados}' não encontrado!")
    
    dataframes_lista = []
    arquivos_processados = []
    
    # Determina quais arquivos buscar
    if formato == 'auto':
        padroes = ['resultados_*.csv', 'resultados_*.parquet']
    elif formato == 'csv':
        padroes = ['resultados_*.csv']
    elif formato == 'parquet':
        padroes = ['resultados_*.parquet']
    else:
        raise ValueError("formato deve ser 'csv', 'parquet' ou 'auto'")
    
    # Busca todos os arquivos
    arquivos = []
    for padrao in padroes:
        caminho_completo = os.path.join(diretorio_resultados, padrao)
        arquivos.extend(glob(caminho_completo))
    
    if not arquivos:
        print(f"⚠ Nenhum arquivo encontrado em '{diretorio_resultados}'")
        return pd.DataFrame()
    
    arquivos = sorted(set(arquivos))  # Remove duplicatas e ordena
    
    print(f"Encontrados {len(arquivos)} arquivos para concatenar")
    print(f"{'='*60}")
    
    for arquivo in arquivos:
        try:
            nome_arquivo = os.path.basename(arquivo)
            
            # Carrega o arquivo apropriado
            if arquivo.endswith('.csv'):
                df_temp = pd.read_csv(arquivo)
                tipo = 'CSV'
            elif arquivo.endswith('.parquet'):
                df_temp = pd.read_parquet(arquivo)
                tipo = 'Parquet'
            else:
                continue
            
            dataframes_lista.append(df_temp)
            arquivos_processados.append(nome_arquivo)
            
            print(f"✓ {nome_arquivo} ({tipo}) - {len(df_temp)} registros")
            
        except Exception as e:
            print(f"✗ ERRO ao carregar {nome_arquivo}: {str(e)[:80]}")
            continue
    
    if not dataframes_lista:
        print("\n⚠ Nenhum DataFrame válido foi carregado!")
        return pd.DataFrame()
    
    print(f"\n{'='*60}")
    print(f"Concatenando {len(dataframes_lista)} DataFrames...")
    
    # Concatena todos os DataFrames
    df_final = pd.concat(dataframes_lista, ignore_index=True)
    
    print(f"✓ Concatenação concluída!")
    print(f"\nRESUMO DO DATASET FINAL:")
    print(f"{'='*60}")
    print(f"  • Total de registros: {len(df_final):,}")
    print(f"  • Total de colunas: {len(df_final.columns)}")
    print(f"  • Anos presentes: {sorted(df_final['year'].unique().tolist())}")
    print(f"  • Registros por ano:")
    
    contagem_por_ano = df_final['year'].value_counts().sort_index()
    for ano, contagem in contagem_por_ano.items():
        print(f"    - {ano}: {contagem:,} patentes")
    
    print(f"{'='*60}")
    
    return df_final


# Função auxiliar para converter as colunas JSON de volta para listas
def converter_json_para_listas(df, prefixo_coluna='Topic_'):
    """
    Converte colunas JSON (strings) de volta para listas Python.
    
    Args:
        df (pd.DataFrame): DataFrame com colunas JSON
        prefixo_coluna (str): Prefixo das colunas de tópicos
    
    Returns:
        pd.DataFrame: DataFrame com colunas convertidas
    """
    df_copia = df.copy()
    
    colunas_topicos = [col for col in df_copia.columns if col.startswith(prefixo_coluna)]
    
    print(f"Convertendo {len(colunas_topicos)} colunas de tópicos...")
    
    for col in colunas_topicos:
        try:
            df_copia[col] = df_copia[col].apply(
                lambda x: json.loads(x) if isinstance(x, str) and x.strip() != '[]' else []
            )
        except Exception as e:
            print(f"⚠ Erro ao converter coluna {col}: {e}")
            continue
    
    print("✓ Conversão concluída!")
    return df_copia

In [22]:
numero_topicos = 5

df_completo = concatenar_resultados_por_ano(
    diretorio_resultados=f'./resultados_por_ano/{numero_topicos}_topics',
    formato='parquet' # Alterar se estiver salvo como CSV
)

if not df_completo.empty:
    
    # Converte as colunas JSON de volta para listas (opcional)
    # os dados ficam em JSON para facilitar o salvamento no formato parquet
    df_completo = converter_json_para_listas(df_completo)
    
    print("\nPROCESSO CONCLUÍDO")
    
    print("\nPreview:")
    print(df_completo.head())
    
else:
    print("\nNenhum dado foi concatenado.")

Encontrados 26 arquivos para concatenar
✓ resultados_1999.parquet (Parquet) - 223 registros
✓ resultados_2000.parquet (Parquet) - 1166 registros
✓ resultados_2001.parquet (Parquet) - 1084 registros
✓ resultados_2002.parquet (Parquet) - 899 registros
✓ resultados_2003.parquet (Parquet) - 764 registros
✓ resultados_2004.parquet (Parquet) - 958 registros
✓ resultados_2005.parquet (Parquet) - 907 registros
✓ resultados_2006.parquet (Parquet) - 800 registros
✓ resultados_2007.parquet (Parquet) - 654 registros
✓ resultados_2008.parquet (Parquet) - 768 registros
✓ resultados_2009.parquet (Parquet) - 603 registros
✓ resultados_2010.parquet (Parquet) - 652 registros
✓ resultados_2011.parquet (Parquet) - 805 registros
✓ resultados_2012.parquet (Parquet) - 394 registros
✓ resultados_2013.parquet (Parquet) - 826 registros
✓ resultados_2014.parquet (Parquet) - 526 registros
✓ resultados_2015.parquet (Parquet) - 901 registros
✓ resultados_2016.parquet (Parquet) - 1275 registros
✓ resultados_2017.par

### Análise das patentes

#### (Não otimizado, utilizado para entender o funcionamento de forma mais clara)

In [11]:
# analise 1

import pandas as pd
import re

df = df_completo.copy()

numero_topicos = 30 # Deve ser igual ao número de tópicos usado no modelo LDA.
numero_palavras_por_topico = 3 # Quantas palavras principais de cada tópico usar.
percentual_similaridade = 0.6 # A similaridade mínima para considerar duas patentes relacionadas.

def extrair_palavras_de_lista(row, topic_cols, n_palavras):
    """
    Extrai as N palavras mais importantes, assumindo que o conteúdo da célula JÁ É UMA LISTA.
    """
    palavras_chave_set = set()
    for col in topic_cols:
        topic_list = row[col]
        
        if not isinstance(topic_list, list):
            print(f"Aviso: Conteúdo da coluna {col} não é uma lista.")
            continue

        palavras_limpas = []
        for item in topic_list:

            if isinstance(item, list) and len(item) > 0:
                palavra_suja = item[0]
                palavra_limpa = re.sub(r"[^a-zA-Zá-úÁ-Ú]", "", palavra_suja)
                if palavra_limpa:
                    palavras_limpas.append(palavra_limpa)
        
        palavras_chave_set.update(palavras_limpas[:n_palavras])
            
    return palavras_chave_set

# Identifica quais colunas são de tópicos
topic_columns = [col for col in df.columns if 'Topic_' in col]

# Cria uma nova coluna com o conjunto de palavras-chave para cada patente
print("Criando a assinatura de palavras para cada patente...")
df['palavras_chave'] = df.apply(
    extrair_palavras_de_lista,
    axis=1,
    args=(topic_columns, numero_palavras_por_topico)
)
print("Assinaturas criadas com sucesso.")

print("\nIniciando a comparação de similaridade entre os anos...")

resultados_finais = []

anos_unicos = sorted(df['year'].unique())

for i in range(len(anos_unicos) - 1):
    ano_atual = anos_unicos[i]
    ano_seguinte = anos_unicos[i+1]
    
    print(f"Comparando ano {ano_atual} com {ano_seguinte}...")
    
    df_ano_atual = df[df['year'] == ano_atual]
    df_ano_seguinte = df[df['year'] == ano_seguinte]

    for _, patente_atual in df_ano_atual.iterrows():
        id_atual = patente_atual['lens_id']
        palavras_atuais = patente_atual['palavras_chave']
        
        patentes_similares_encontradas = []
        
        for _, patente_seguinte in df_ano_seguinte.iterrows():
            id_seguinte = patente_seguinte['lens_id']
            palavras_seguintes = patente_seguinte['palavras_chave']
            
            # Calcula a similaridade (Coeficiente de Jaccard)
            # Jaccard = (tamanho da interseção) / (tamanho da união)
            intersecao = len(palavras_atuais.intersection(palavras_seguintes))
            uniao = len(palavras_atuais.union(palavras_seguintes))
            
            if uniao == 0:
                similaridade = 0
            else:
                similaridade = intersecao / uniao
            
            # Verifica se a similaridade atinge o nosso limite
            if similaridade >= percentual_similaridade:
                patentes_similares_encontradas.append(id_seguinte)
        
        # Guarda o resultado para a patente atual
        resultados_finais.append({
            'lens_id': id_atual,
            'patentes_similares': patentes_similares_encontradas
        })

print("Comparação finalizada.")

# Cria um DataFrame com os resultados
df_resultados = pd.DataFrame(resultados_finais)

# Junta os resultados ao DataFrame original usando o 'lens_id' como chave
df = pd.merge(df, df_resultados, on='lens_id', how='left')

# Preenche com uma lista vazia as patentes que não tiveram correspondência (as do último ano)
df['patentes_similares'] = df['patentes_similares'].apply(lambda x: x if isinstance(x, list) else [])

# Cria as duas colunas finais
df['count_similares'] = df['patentes_similares'].apply(len)
df['emergente'] = df['count_similares'] > 0

print("\nResultado Final:")
display(df[['lens_id', 'year', 'patentes_similares', 'count_similares', 'emergente']])

df.to_csv(f'./resultados/{numero_topicos}_topics/analise-de-similaridade_n{numero_palavras_por_topico}_s{percentual_similaridade:.2f}.csv', index=True)

Criando a assinatura de palavras para cada patente...


KeyboardInterrupt: 

In [12]:
# analise 2

import math

df2 = df_completo.copy()

metodo = 'tanimoto'  # 'cossenos' ou 'tanimoto'
numero_topicos = 30
numero_palavras_por_topico = 3 
percentual_similaridade = 0.6

def extrair_vetor_de_caracteristicas(row, topic_cols, n_palavras):
    """
    Extrai um dicionário {palavra: probabilidade} para uma patente, 
    que servirá como seu vetor de características.
    """
    vetor_caracteristicas = {}
    for col in topic_cols:
        topic_list = row[col]
        
        if not isinstance(topic_list, list):
            continue

        palavras_com_peso = []
        for item in topic_list:
            # Garante que o item é uma lista com [palavra, probabilidade]
            if isinstance(item, list) and len(item) == 2:
                palavra_suja = item[0]
                probabilidade = item[1]
                
                palavra_limpa = re.sub(r"[^a-zA-Zá-úÁ-Ú]", "", str(palavra_suja))
                
                if palavra_limpa and isinstance(probabilidade, (int, float)):
                    palavras_com_peso.append((palavra_limpa, probabilidade))
        
        # Adiciona as 'n_palavras' mais importantes ao nosso dicionário
        for palavra, prob in palavras_com_peso[:n_palavras]:
             # Se a palavra já existe (veio de outro tópico), somamos as probabilidades
             # ou podemos simplesmente manter a mais alta. Vamos mantê-la simples e usar a primeira que aparecer.
             if palavra not in vetor_caracteristicas:
                vetor_caracteristicas[palavra] = prob
                
    return vetor_caracteristicas

def calcular_similaridade_cossenos(vetor1, vetor2):
    """
    Calcula a similaridade de cossenos entre dois dicionários {palavra: prob}.
    """
    # Encontra as palavras que são comuns a ambos os vetores
    palavras_comuns = set(vetor1.keys()).intersection(set(vetor2.keys()))
    
    # Se não houver palavras em comum, a similaridade é 0
    if not palavras_comuns:
        return 0.0

    # 1. Calcula o produto escalar (dot product)
    # Soma da multiplicação das probabilidades das palavras em comum
    produto_escalar = sum(vetor1[palavra] * vetor2[palavra] for palavra in palavras_comuns)

    # 2. Calcula a magnitude (norma) de cada vetor
    magnitude_vetor1 = math.sqrt(sum(prob**2 for prob in vetor1.values()))
    magnitude_vetor2 = math.sqrt(sum(prob**2 for prob in vetor2.values()))
    
    # Se alguma magnitude for zero, evita divisão por zero
    if magnitude_vetor1 == 0 or magnitude_vetor2 == 0:
        return 0.0

    # 3. Calcula a similaridade
    similaridade = produto_escalar / (magnitude_vetor1 * magnitude_vetor2)
    
    return similaridade

def calcular_similaridade_tanimoto(vetor1, vetor2):
    """
    Calcula a similaridade de Tanimoto (Jaccard Generalizado) entre
    dois dicionários {palavra: prob}, que considera a magnitude.
    O resultado é normalizado entre 0 e 1.
    """
    
    # 1. Encontra as palavras que são comuns a ambos os vetores
    # (O Produto Escalar só se aplica a elas)
    palavras_comuns = set(vetor1.keys()).intersection(set(vetor2.keys()))

    # 2. Calcula o Produto Escalar (A . B)
    # Soma da multiplicação das probabilidades das palavras em comum
    produto_escalar = sum(vetor1[palavra] * vetor2[palavra] for palavra in palavras_comuns)
    
    # Se não há sobreposição, a similaridade é 0
    if produto_escalar == 0.0:
        return 0.0

    # 3. Calcula o quadrado da magnitude de cada vetor (|A|^2 e |B|^2)
    # Nota: Não precisamos da raiz quadrada (sqrt) aqui,
    # a fórmula usa a magnitude ao quadrado.
    mag_quad_vetor1 = sum(prob**2 for prob in vetor1.values())
    mag_quad_vetor2 = sum(prob**2 for prob in vetor2.values())
    
    # 4. Calcula o denominador da fórmula de Tanimoto
    denominador = mag_quad_vetor1 + mag_quad_vetor2 - produto_escalar
    
    # Evita divisão por zero
    if denominador == 0:
        # Isso só aconteceria se ambos os vetores fossem nulos
        return 0.0

    # 5. Calcula a similaridade
    similaridade = produto_escalar / denominador
    
    return similaridade

topic_columns = [col for col in df2.columns if 'Topic_' in col]

print("Criando os vetores de características para cada patente...")
df2['vetor_caracteristicas'] = df2.apply(
    extrair_vetor_de_caracteristicas,
    axis=1,
    args=(topic_columns, numero_palavras_por_topico)
)
print("Vetores criados com sucesso.")

print("\nIniciando a comparação de similaridade (Cossenos) entre os anos...")

resultados_finais = []
anos_unicos = sorted(df2['year'].unique())

for i in range(len(anos_unicos) - 1):
    ano_atual = anos_unicos[i]
    ano_seguinte = anos_unicos[i+1]
    
    print(f"Comparando ano {ano_atual} com {ano_seguinte}...")
    
    df_ano_atual = df2[df2['year'] == ano_atual]
    df_ano_seguinte = df2[df2['year'] == ano_seguinte]

    for _, patente_atual in df_ano_atual.iterrows():
        id_atual = patente_atual['lens_id']
        vetor_atual = patente_atual['vetor_caracteristicas'] # <-- Usa o vetor
        
        patentes_similares_encontradas = []
        
        for _, patente_seguinte in df_ano_seguinte.iterrows():
            id_seguinte = patente_seguinte['lens_id']
            vetor_seguinte = patente_seguinte['vetor_caracteristicas'] # <-- Usa o vetor
            
            if metodo == 'cossenos': 
                similaridade = calcular_similaridade_cossenos(vetor_atual, vetor_seguinte)
            elif metodo == 'tanimoto':
                similaridade = calcular_similaridade_tanimoto(vetor_atual, vetor_seguinte)
            else:
                raise ValueError("Método desconhecido para similaridade.")
            
            if similaridade >= percentual_similaridade:
                patentes_similares_encontradas.append(id_seguinte)
        
        resultados_finais.append({
            'lens_id': id_atual,
            'patentes_similares': patentes_similares_encontradas
        })

print("Comparação finalizada.")

# O resto do código para juntar os resultados é exatamente o mesmo
df_resultados = pd.DataFrame(resultados_finais)
df2 = pd.merge(df2, df_resultados, on='lens_id', how='left')
df2['patentes_similares'] = df2['patentes_similares'].apply(lambda x: x if isinstance(x, list) else [])
df2['count_similares'] = df2['patentes_similares'].apply(len)
df2['emergente'] = df2['count_similares'] > 0

print("\nResultado Final:")
display(df2[['lens_id', 'year', 'patentes_similares', 'count_similares', 'emergente']])

df2.to_csv(f'./resultados/{numero_topicos}_topics/analise-de-similaridade2_n{numero_palavras_por_topico}_s{percentual_similaridade:.2f}.csv', index=True)

Criando os vetores de características para cada patente...


KeyboardInterrupt: 

### Análise das patentes (Otimizado)

In [23]:
# definir o número de tópicos em gerar dataframe com resultados
numero_palavras_por_topico = 5 # Quantas palavras principais de cada tópico usar.
percentual_similaridade = 0.95 # A similaridade mínima para considerar duas patentes relacionadas.

In [24]:
# Codigo que tem a mesma função da (analise 1), mas otimizado para performance
# Otimizado pelo Gemini, não modifiquei nada
# ele altera levemente a lógica de limpeza das palavras mudando a ordem delas,
# mas o resultado final é equivalente

import pandas as pd
import re
from collections import defaultdict, Counter

df_otimizado = df_completo.copy()

# numero_palavras_por_topico = 3
# percentual_similaridade = 0.90

# Identifica quais colunas são de tópicos
topic_columns = [col for col in df_otimizado.columns if 'Topic_' in col]

# --- Parte 2: Otimização 1 - Extração Vetorizada de Palavras-Chave ---
# Esta parte substitui a função 'extrair_palavras_de_lista' e o 'df_otimizado.apply'

print("Iniciando extração vetorizada de palavras-chave...")

# 1. 'Melt' transforma colunas (Topic_1, Topic_2) em linhas
#    Isso nos dá uma estrutura longa, mais fácil de processar.
df_melted = df_otimizado.melt(
    id_vars=['lens_id'], 
    value_vars=topic_columns, 
    value_name='topic_list'
)

# 2. Remove linhas onde não havia dados de tópico
df_melted = df_melted.dropna(subset=['topic_list'])

# 3. 'Explode' transforma listas em linhas
#    Se 'topic_list' era [['word1', 0.5], ['word2', 0.4]],
#    agora teremos duas linhas: ['word1', 0.5] e ['word2', 0.4]
df_exploded = df_melted.explode('topic_list')

# 4. Filtra itens que não são listas válidas (como esperado pela função original)
df_exploded = df_exploded[
    df_exploded['topic_list'].apply(lambda x: isinstance(x, list) and len(x) > 0)
]

# 5. Extrai a palavra (primeiro item da sub-lista) de forma vetorizada
df_exploded['palavra_suja'] = df_exploded['topic_list'].str[0]

# 6. Limpa a palavra de forma vetorizada (usando expressões regulares)

def clean_word_original(palavra_suja):
    if isinstance(palavra_suja, str):
        palavra_limpa = re.sub(r"[^a-zA-Zá-úÁ-Ú]", "", palavra_suja)
        # A lógica original `if palavra_limpa:` filtrava strings vazias.
        # Retornar None para strings vazias tem o mesmo efeito no passo 7.
        return palavra_limpa if palavra_limpa else None
    # Se não for uma string (ex: None, 123), trata como inválido
    return None

print("Limpando palavras-chave...")
df_exploded['palavra_limpa'] = df_exploded['palavra_suja'].apply(clean_word_original)

# 7. Remove palavras que ficaram vazias após a limpeza
df_exploded = df_exploded.dropna(subset=['palavra_limpa'])

# 8. Agrupa por patente E por tópico de origem, e pega as N primeiras
#    Isso simula o '[:n_palavras]' da função original, para CADA tópico
df_top_n = df_exploded.groupby(['lens_id', 'variable']).head(numero_palavras_por_topico)

# 9. Finalmente, agrupa por patente (lens_id) e coleta todas as palavras únicas em um set
palavras_chave_series = df_top_n.groupby('lens_id')['palavra_limpa'].apply(set)

# 10. Mapeia os 'sets' de palavras-chave de volta ao DataFrame original
df_otimizado['palavras_chave'] = df_otimizado['lens_id'].map(palavras_chave_series)

# Preenche com 'sets' vazios os IDs que não tinham tópicos
df_otimizado['palavras_chave'] = df_otimizado['palavras_chave'].apply(lambda x: x if isinstance(x, set) else set())

print("Assinaturas de palavras criadas com sucesso.")

# --- Parte 3: Otimização 2 - Comparação de Similaridade com Índice Invertido ---

print("\nIniciando a comparação de similaridade otimizada...")

resultados_finais = []
anos_unicos = sorted(df_otimizado['year'].unique())

for i in range(len(anos_unicos) - 1):
    ano_atual = anos_unicos[i]
    ano_seguinte = anos_unicos[i+1]
    
    print(f"Comparando ano {ano_atual} com {ano_seguinte}...")
    
    # Filtra os DataFrames por ano
    df_ano_atual = df_otimizado[df_otimizado['year'] == ano_atual]
    df_ano_seguinte = df_otimizado[df_otimizado['year'] == ano_seguinte]

    # --- Otimização Principal ---
    
    # 1. Cria um mapa de lookup rápido para o ano seguinte: {lens_id -> set_de_palavras}
    #    Isso é muito mais rápido que filtrar o DataFrame dentro de um loop
    palavras_seguintes_map = df_ano_seguinte.set_index('lens_id')['palavras_chave']

    # 2. Cria o Índice Invertido para o ano seguinte
    #    Formato: {palavra -> {id1, id2, ...}}
    inverted_index = defaultdict(set)
    for lens_id, palavras_set in palavras_seguintes_map.items():
        for palavra in palavras_set:
            inverted_index[palavra].add(lens_id)
            
    # 3. Itera sobre o ano atual (usando itertuples() que é mais rápido que iterrows())
    for patente_atual in df_ano_atual.itertuples():
        id_atual = patente_atual.lens_id
        palavras_atuais = patente_atual.palavras_chave
        len_atuais = len(palavras_atuais) # Pré-calcula o tamanho
        
        if len_atuais == 0: # Pula patentes sem palavras-chave
            resultados_finais.append({
                'lens_id': id_atual,
                'patentes_similares': []
            })
            continue

        # 4. Encontra candidatos usando o Índice Invertido
        #    'candidate_counts' irá armazenar: {id_candidato -> contagem_de_palavras_em_comum}
        #    A contagem de palavras em comum é exatamente o tamanho da INTERSEÇÃO
        candidate_counts = Counter()
        for palavra in palavras_atuais:
            # O 'update' do Counter soma as contagens, mas como o índice
            # tem 'sets' (ids únicos por palavra), usamos 'inverted_index[palavra]'
            # para adicionar 1 a cada ID que contém essa palavra.
            candidate_counts.update(inverted_index[palavra])
            
        patentes_similares_encontradas = []
        
        # 5. Calcula a similaridade APENAS para os candidatos
        for id_seguinte, intersecao in candidate_counts.items():
            
            # Não precisamos mais calcular a interseção, já a temos!
            
            # Pega os dados do candidato no mapa de lookup
            palavras_seguintes = palavras_seguintes_map[id_seguinte]
            
            # Calcula a união: (tam_A + tam_B - intersecao)
            uniao = len_atuais + len(palavras_seguintes) - intersecao
            
            if uniao == 0:
                similaridade = 0
            else:
                similaridade = intersecao / uniao
            
            # 6. Verifica se a similaridade atinge o nosso limite
            if similaridade >= percentual_similaridade:
                patentes_similares_encontradas.append(id_seguinte)
        
        # Guarda o resultado para a patente atual
        resultados_finais.append({
            'lens_id': id_atual,
            'patentes_similares': patentes_similares_encontradas
        })

print("Comparação finalizada.")

# --- Parte 4: Pós-processamento e Exportação (Igual ao original) ---

# Cria um DataFrame com os resultados
df_resultados = pd.DataFrame(resultados_finais)

# Junta os resultados ao DataFrame original usando o 'lens_id' como chave
df_otimizado = pd.merge(df_otimizado, df_resultados, on='lens_id', how='left')

# Preenche com uma lista vazia as patentes que não tiveram correspondência
# (as do último ano ou as sem correspondência)
# A lógica original estava incompleta, esta é mais segura:
df_otimizado['patentes_similares'] = df_otimizado['patentes_similares'].apply(lambda x: x if isinstance(x, list) else [])

# Cria as duas colunas finais
df_otimizado['count_similares'] = df_otimizado['patentes_similares'].apply(len)
df_otimizado['emergente'] = df_otimizado['count_similares'] > 0

print("\nResultado Final:")
display(df_otimizado[['lens_id', 'year', 'patentes_similares', 'count_similares', 'emergente']])

# Salva o arquivo final
os.makedirs(f'./resultados_analise/{numero_topicos}_topics/', exist_ok=True)
df_otimizado.to_csv(f'./resultados_analise/{numero_topicos}_topics/analise-de-similaridade_n{numero_palavras_por_topico}_s{percentual_similaridade:.2f}.csv', index=True)

print("\nArquivo salvo com sucesso.")

Iniciando extração vetorizada de palavras-chave...
Limpando palavras-chave...
Assinaturas de palavras criadas com sucesso.

Iniciando a comparação de similaridade otimizada...
Comparando ano 1999 com 2000...
Comparando ano 2000 com 2001...
Comparando ano 2001 com 2002...
Comparando ano 2002 com 2003...
Comparando ano 2003 com 2004...
Comparando ano 2004 com 2005...
Comparando ano 2005 com 2006...
Comparando ano 2006 com 2007...
Comparando ano 2007 com 2008...
Comparando ano 2008 com 2009...
Comparando ano 2009 com 2010...
Comparando ano 2010 com 2011...
Comparando ano 2011 com 2012...
Comparando ano 2012 com 2013...
Comparando ano 2013 com 2014...
Comparando ano 2014 com 2015...
Comparando ano 2015 com 2016...
Comparando ano 2016 com 2017...
Comparando ano 2017 com 2018...
Comparando ano 2018 com 2019...
Comparando ano 2019 com 2020...
Comparando ano 2020 com 2021...
Comparando ano 2021 com 2022...
Comparando ano 2022 com 2023...
Comparando ano 2023 com 2024...
Comparação finalizada.



,lens_id,year,patentes_similares,count_similares,emergente
0,171-098-512-075-612,1999,[],0,False
1,046-440-553-666-653,1999,[],0,False
2,157-865-805-179-279,1999,[],0,False
3,104-140-108-095-175,1999,[],0,False
4,023-683-667-869-67X,1999,[],0,False
...,...,...,...,...,...
25235,114-648-369-763-290,2024,[],0,False
25236,154-085-879-471-233,2024,[],0,False
25237,052-411-343-097-485,2024,[],0,False
25238,056-089-956-710-61X,2024,[],0,False



Arquivo salvo com sucesso.


In [25]:
# Codigo que tem a mesma função da (analise 2), mas otimizado para performance
# Otimizado pelo Gemini, não modifiquei nada

from collections import defaultdict, Counter

df2 = df_completo.copy()

metodo = 'tanimoto'  # 'cossenos' ou 'tanimoto'
# numero_palavras_por_topico = 3 
# percentual_similaridade = 0.90

# Identifica quais colunas são de tópicos
topic_columns = [col for col in df2.columns if 'Topic_' in col]

# --- Parte 2: Otimização 1 - Extração Vetorizada de Vetores de Características ---
print("Iniciando extração vetorizada de vetores de características...")

# 1. 'Melt' transforma colunas (Topic_1, Topic_2) em linhas
df_melted = df2.melt(
    id_vars=['lens_id'], 
    value_vars=topic_columns, 
    var_name='topic_source',  # Guarda o nome da coluna de tópico original
    value_name='topic_list'
)

# 2. Remove linhas onde não havia dados de tópico
df_melted = df_melted.dropna(subset=['topic_list'])

# 3. 'Explode' transforma listas em linhas
df_exploded = df_melted.explode('topic_list')
df_exploded = df_exploded.dropna(subset=['topic_list'])

# 4. Valida o formato [palavra, prob]
#    Usamos 'apply' aqui, mas é em menos dados e mais complexo de vetorizar
df_exploded['is_valid'] = df_exploded['topic_list'].apply(
    lambda x: isinstance(x, list) and len(x) == 2
)
df_exploded = df_exploded[df_exploded['is_valid']]

# 5. Extrai palavra e probabilidade de forma vetorizada
df_exploded['palavra_suja'] = df_exploded['topic_list'].str[0].astype(str)
df_exploded['probabilidade'] = pd.to_numeric(
    df_exploded['topic_list'].str[1], errors='coerce'
)

# 6. Remove linhas que não tinham probabilidades numéricas
df_exploded = df_exploded.dropna(subset=['probabilidade'])

# 7. Limpa a palavra de forma vetorizada
df_exploded['palavra_limpa'] = df_exploded['palavra_suja'].str.replace(
    r"[^a-zA-Zá-úÁ-Ú]", "", regex=True
)

# 8. Remove palavras que ficaram vazias
df_exploded = df_exploded[df_exploded['palavra_limpa'] != '']

# 9. Agrupa por patente E por tópico de origem, e pega as N primeiras
#    Isso simula o '[:n_palavras]' da função original, para CADA tópico
df_top_n = df_exploded.groupby(['lens_id', 'topic_source']).head(numero_palavras_por_topico)

# 10. Lida com a lógica 'if palavra not in vetor_caracteristicas:'
#     Ao remover duplicatas por 'lens_id' e 'palavra_limpa', mantendo a 
#     primeira ocorrência, simulamos essa lógica.
df_final_palavras = df_top_n.drop_duplicates(subset=['lens_id', 'palavra_limpa'], keep='first')

# 11. Finalmente, agrupa por patente e cria o dicionário (vetor)
vetores_series = df_final_palavras.groupby('lens_id').apply(
    lambda x: dict(zip(x['palavra_limpa'], x['probabilidade']))
)

# 12. Mapeia os vetores de volta ao DataFrame original
df2['vetor_caracteristicas'] = df2['lens_id'].map(vetores_series)
df2['vetor_caracteristicas'] = df2['vetor_caracteristicas'].apply(
    lambda x: x if isinstance(x, dict) else {}
)

print("Vetores criados com sucesso.")

del df_melted
del df_exploded
del df_top_n
del df_final_palavras
del vetores_series

# --- Parte 3: Otimização 2 - Pré-cálculo das Magnitudes ---
print(f"Pré-calculando magnitudes para o método '{metodo}'...")

if metodo == 'cossenos':
    # Para Cossenos, precisamos da magnitude (raiz quadrada da soma dos quadrados)
    df2['magnitude'] = df2['vetor_caracteristicas'].apply(
        lambda v: math.sqrt(sum(prob**2 for prob in v.values()))
    )
elif metodo == 'tanimoto':
    # Para Tanimoto, precisamos da magnitude ao quadrado (soma dos quadrados)
    df2['mag_quadrada'] = df2['vetor_caracteristicas'].apply(
        lambda v: sum(prob**2 for prob in v.values())
    )
else:
    raise ValueError("Método desconhecido para similaridade.")

print("Magnitudes calculadas.")

# --- Parte 4: Otimização 3 - Comparação com Índice Invertido Ponderado ---

print(f"\nIniciando a comparação de similaridade ({metodo}) otimizada...")

resultados_finais = []
anos_unicos = sorted(df2['year'].unique())

for i in range(len(anos_unicos) - 1):
    ano_atual = anos_unicos[i]
    ano_seguinte = anos_unicos[i+1]
    
    print(f"Comparando ano {ano_atual} com {ano_seguinte}...")
    
    # Filtra os DataFrames por ano
    df_ano_atual = df2[df2['year'] == ano_atual]
    df_ano_seguinte = df2[df2['year'] == ano_seguinte]

    # --- Otimização Principal ---
    
    # 1. Cria mapas de lookup rápidos para o ano seguinte
    vetores_seguintes_map = df_ano_seguinte.set_index('lens_id')['vetor_caracteristicas']
    
    if metodo == 'cossenos':
        mag_seguintes_map = df_ano_seguinte.set_index('lens_id')['magnitude']
    else: # tanimoto
        mag_seguintes_map = df_ano_seguinte.set_index('lens_id')['mag_quadrada']

    # 2. Cria o Índice Invertido Ponderado para o ano seguinte
    #    Formato: {palavra -> {id1: prob1, id2: prob2, ...}}
    inverted_index = defaultdict(dict)
    for lens_id, vetor in vetores_seguintes_map.items():
        for palavra, prob in vetor.items():
            inverted_index[palavra][lens_id] = prob
            
    # 3. Itera sobre o ano atual (usando itertuples() que é mais rápido)
    for patente_atual in df_ano_atual.itertuples():
        id_atual = patente_atual.lens_id
        vetor_atual = patente_atual.vetor_caracteristicas
        
        # Pula patentes sem vetor
        if not vetor_atual:
            resultados_finais.append({'lens_id': id_atual, 'patentes_similares': []})
            continue

        # 4. Calcula os produtos escalares para TODOS os candidatos de uma vez
        #    'dot_products' armazenará: {id_candidato -> produto_escalar}
        dot_products = Counter()
        for palavra, prob_atual in vetor_atual.items():
            # Se a palavra existe no índice do ano seguinte...
            if palavra in inverted_index:
                # ...itera sobre os candidatos que a possuem
                for id_seguinte, prob_seguinte in inverted_index[palavra].items():
                    # Acumula o produto escalar
                    dot_products[id_seguinte] += prob_atual * prob_seguinte
            
        patentes_similares_encontradas = []
        
        # 5. Calcula a similaridade APENAS para os candidatos (onde dot_product > 0)
        if metodo == 'cossenos':
            mag_atual = patente_atual.magnitude
            if mag_atual == 0: continue # Pula se a magnitude for 0
            
            for id_seguinte, produto_escalar in dot_products.items():
                mag_seguinte = mag_seguintes_map[id_seguinte]
                if mag_seguinte == 0: continue
                
                similaridade = produto_escalar / (mag_atual * mag_seguinte)
                if similaridade >= percentual_similaridade:
                    patentes_similares_encontradas.append(id_seguinte)
        
        elif metodo == 'tanimoto':
            mag_sq_atual = patente_atual.mag_quadrada
            
            for id_seguinte, produto_escalar in dot_products.items():
                mag_sq_seguinte = mag_seguintes_map[id_seguinte]
                
                denominador = mag_sq_atual + mag_sq_seguinte - produto_escalar
                if denominador == 0: continue
                
                similaridade = produto_escalar / denominador
                if similaridade >= percentual_similaridade:
                    patentes_similares_encontradas.append(id_seguinte)
        
        # Guarda o resultado para a patente atual
        resultados_finais.append({
            'lens_id': id_atual,
            'patentes_similares': patentes_similares_encontradas
        })

print("Comparação finalizada.")

# --- Parte 5: Pós-processamento e Exportação (Igual ao original) ---

df_resultados = pd.DataFrame(resultados_finais)
df2 = pd.merge(df2, df_resultados, on='lens_id', how='left')
df2['patentes_similares'] = df2['patentes_similares'].apply(lambda x: x if isinstance(x, list) else [])
df2['count_similares'] = df2['patentes_similares'].apply(len)
df2['emergente'] = df2['count_similares'] > 0

# (Remove as colunas auxiliares de magnitude se não as quiser no CSV final)
if 'magnitude' in df2.columns:
    df2 = df2.drop(columns=['magnitude'])
if 'mag_quadrada' in df2.columns:
    df2 = df2.drop(columns=['mag_quadrada'])

print("\nResultado Final:")
display(df2[['lens_id', 'year', 'patentes_similares', 'count_similares', 'emergente']])

# Salva o arquivo final
df2.to_csv(f'./resultados_analise/{numero_topicos}_topics/analise-de-similaridade2_{metodo}_n{numero_palavras_por_topico}_s{percentual_similaridade:.2f}.csv', index=True)

print("\nArquivo salvo com sucesso.")

Iniciando extração vetorizada de vetores de características...


C:\Users\pedro\AppData\Local\Temp\ipykernel_5264\2297605742.py:67: FutureWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  vetores_series = df_final_palavras.groupby('lens_id').apply(


Vetores criados com sucesso.
Pré-calculando magnitudes para o método 'tanimoto'...
Magnitudes calculadas.

Iniciando a comparação de similaridade (tanimoto) otimizada...
Comparando ano 1999 com 2000...
Comparando ano 2000 com 2001...
Comparando ano 2001 com 2002...
Comparando ano 2002 com 2003...
Comparando ano 2003 com 2004...
Comparando ano 2004 com 2005...
Comparando ano 2005 com 2006...
Comparando ano 2006 com 2007...
Comparando ano 2007 com 2008...
Comparando ano 2008 com 2009...
Comparando ano 2009 com 2010...
Comparando ano 2010 com 2011...
Comparando ano 2011 com 2012...
Comparando ano 2012 com 2013...
Comparando ano 2013 com 2014...
Comparando ano 2014 com 2015...
Comparando ano 2015 com 2016...
Comparando ano 2016 com 2017...
Comparando ano 2017 com 2018...
Comparando ano 2018 com 2019...
Comparando ano 2019 com 2020...
Comparando ano 2020 com 2021...
Comparando ano 2021 com 2022...
Comparando ano 2022 com 2023...
Comparando ano 2023 com 2024...
Comparação finalizada.

Result

,lens_id,year,patentes_similares,count_similares,emergente
0,171-098-512-075-612,1999,"[116-121-940-543-96X, 197-447-606-499-762, 169...",9,True
1,046-440-553-666-653,1999,[],0,False
2,157-865-805-179-279,1999,[],0,False
3,104-140-108-095-175,1999,[],0,False
4,023-683-667-869-67X,1999,[],0,False
...,...,...,...,...,...
25235,114-648-369-763-290,2024,[],0,False
25236,154-085-879-471-233,2024,[],0,False
25237,052-411-343-097-485,2024,[],0,False
25238,056-089-956-710-61X,2024,[],0,False



Arquivo salvo com sucesso.


### Carregar dataframe com os resultados

In [26]:
numero_topicos = 5
numero_palavras_por_topico = 5
percentual_similaridade = 0.95

caminho_do_arquivo = f'./resultados_analise/{numero_topicos}_topics/analise-de-similaridade_n{numero_palavras_por_topico}_s{percentual_similaridade:.2f}.csv'
caminho_do_arquivo2 = f'./resultados_analise/{numero_topicos}_topics/analise-de-similaridade2_tanimoto_n{numero_palavras_por_topico}_s{percentual_similaridade:.2f}.csv'

In [27]:
# carrega o csv da primeira analise

import pandas as pd
import ast

def parse_set_safe(val):
    """
    Converte uma string "{'a', 'b'}" num set {'a', 'b'}.
    Retorna um set() vazio se a entrada for NaN ou inválida.
    """
    if pd.isna(val):
        return set()
    try:
        # ast.literal_eval é a forma SEGURA de avaliar um literal Python
        result = ast.literal_eval(val)
        if isinstance(result, set):
            return result
        # Se for outro tipo (ex: lista), converte para set
        return set(result) 
    except (ValueError, SyntaxError, TypeError):
        # Se falhar (string malformada), retorna set vazio
        return set()

def parse_list_safe(val):
    """
    Converte uma string "['a', 'b']" numa lista ['a', 'b'].
    Retorna uma list() vazia se a entrada for NaN ou inválida.
    """
    if pd.isna(val):
        return []
    try:
        # ast.literal_eval é a forma SEGURA de avaliar um literal Python
        result = ast.literal_eval(val)
        if isinstance(result, list):
            return result
        # Se for outro tipo (ex: set), converte para lista
        return list(result)
    except (ValueError, SyntaxError, TypeError):
        # Se falhar (string malformada), retorna lista vazia
        return []


# Dicionário de 'converters'
# Mapeia o nome da coluna para a função que deve ser usada nela
converters_dict = {
    'palavras_chave': parse_set_safe,       # Converte esta coluna para SETs
    'patentes_similares': parse_list_safe   # Converte esta coluna para LISTAs
}

# Dicionário de 'dtypes' (Tipos de Dados)
# Para forçar o pandas a usar os tipos corretos e mais eficientes
dtype_dict = {
    'lens_id': 'string',  # Mais eficiente que 'object'
    'year': 'Int64',      # 'Int64' (com 'I' maiúsculo) permite valores nulos (NaN)
    'count_similares': 'Int64',
    'emergente': 'boolean'  # 'boolean' (minúsculo) permite valores nulos (NA)
}

print(f"Carregando o arquivo: {caminho_do_arquivo}...")

try:
    # Carrega o DataFrame aplicando todas as nossas regras
    df_carregado = pd.read_csv(
        caminho_do_arquivo,
        index_col=0,           # A primeira coluna (sem nome) é o índice
        converters=converters_dict,  # Aplica os conversores de set/list
        dtype=dtype_dict,            # Força os tipos de dados corretos
        low_memory=False       # Recomendado para arquivos complexos
    )

    print("\nArquivo carregado com sucesso!")
    print("Verificando os tipos de dados das colunas:")
    
    # Verifica os tipos (dtypes)
    print(df_carregado.dtypes)
    
    print("\nExemplo de dados convertidos (primeira linha):")
    # Mostra um exemplo dos dados convertidos
    print(f"Tipo 'palavras_chave': {type(df_carregado.iloc[0]['palavras_chave'])}")
    print(f"Tipo 'patentes_similares': {type(df_carregado.iloc[0]['patentes_similares'])}")

    # Mostra o DataFrame carregado
    print("\nCabeçalho do DataFrame Carregado:")
    display(df_carregado.head())

    # A variável 'df_carregado' está pronta para ser usada!
    # Por exemplo, pode atribuí-la a 'df_completo' para os próximos scripts
    # df_completo = df_carregado.copy()

except FileNotFoundError:
    print(f"ERRO: O arquivo '{caminho_do_arquivo}' não foi encontrado.")
except Exception as e:
    print(f"Ocorreu um erro inesperado durante o carregamento: {e}")

Carregando o arquivo: ./resultados_analise/5_topics/analise-de-similaridade_n5_s0.95.csv...

Arquivo carregado com sucesso!
Verificando os tipos de dados das colunas:
lens_id               string[python]
year                           Int64
Topic_0                       object
Topic_1                       object
Topic_2                       object
Topic_3                       object
Topic_4                       object
palavras_chave                object
patentes_similares            object
count_similares                Int64
emergente                    boolean
dtype: object

Exemplo de dados convertidos (primeira linha):
Tipo 'palavras_chave': <class 'set'>
Tipo 'patentes_similares': <class 'list'>

Cabeçalho do DataFrame Carregado:


,lens_id,year,Topic_0,Topic_1,Topic_2,Topic_3,Topic_4,palavras_chave,patentes_similares,count_similares,emergente
0,171-098-512-075-612,1999,"[[""'estrutura',"", 3.898665454471484e-06], [""'d...","[[""'invenção',"", 9.288967703469098e-06], [""'pa...","[[""'superfície',"", 6.86711155140074e-06], [""'p...","[[""'invenção',"", 6.183001005410915e-06], [""'di...","[[""'invenção',"", 0.009545245207846165], [""'est...","{superfície, apresentar, invenção, patente, mo...",[],0,False
1,046-440-553-666-653,1999,"[[""'outro',"", 4.0580184759164695e-06], [""'pate...","[[""'invenção',"", 8.62777051224839e-06], [""'pat...","[[""'patente',"", 4.253686711308546e-06], [""'inv...","[[""'invenção',"", 0.007665595971047878], [""'fol...","[[""'invenção',"", 8.391961273446213e-06], [""'pa...","{exterior, invenção, patente, outro, folha, cé...",[],0,False
2,157-865-805-179-279,1999,"[[""'movimento',"", 2.9853720207029255e-06], [""'...","[[""'invenção',"", 5.626234269584529e-06], [""'co...","[[""'superfície',"", 4.159340278420132e-06], [""'...","[[""'invenção',"", 0.0076750656589865685], [""'di...","[[""'invenção',"", 5.4724605433875695e-06], [""'e...","{superfície, invenção, presente, patente, dire...",[],0,False
3,104-140-108-095-175,1999,"[[""'porta',"", 0.038128580898046494], [""'movime...","[[""'invenção',"", 6.123864295659587e-06], [""'el...","[[""'lateral',"", 3.751218400793732e-06], [""'ele...","[[""'elemento',"", 4.126146905036876e-06], [""'in...","[[""'construção',"", 7.65810636949027e-06], [""'e...","{chapa, invenção, patente, direção, fixação, p...",[],0,False
4,023-683-667-869-67X,1999,"[[""'água',"", 2.3737948140478693e-06], [""'conju...","[[""'invenção',"", 4.9905688683793414e-06], [""'m...","[[""'água',"", 4.833969342143973e-06], [""'formar...","[[""'invenção',"", 0.007677071262151003], [""'fol...","[[""'invenção',"", 4.854169674217701e-06], [""'pa...","{material, invenção, aparelho, água, patente, ...",[],0,False


In [28]:
import pandas as pd
import ast

def parse_dict_safe(val):
    """
    Converte uma string "{'a': 1.0}" num dicionário {'a': 1.0}.
    Retorna um dict() vazio se a entrada for NaN ou inválida.
    """
    if pd.isna(val):
        return {}
    try:
        # ast.literal_eval é a forma SEGURA de avaliar um literal Python
        result = ast.literal_eval(val)
        if isinstance(result, dict):
            return result
        # Se falhar a verificação, retorna vazio
        return {}
    except (ValueError, SyntaxError, TypeError):
        # Se falhar a conversão (string malformada), retorna vazio
        return {}

def parse_list_safe(val):
    """
    Converte uma string "['a', 'b']" numa lista ['a', 'b'].
    Retorna uma list() vazia se a entrada for NaN ou inválida.
    """
    if pd.isna(val):
        return []
    try:
        result = ast.literal_eval(val)
        if isinstance(result, list):
            return result
        return list(result)
    except (ValueError, SyntaxError, TypeError):
        return []


# Dicionário de 'converters'
# Mapeia o nome da coluna para a função que deve ser usada nela
converters_dict = {
    'vetor_caracteristicas': parse_dict_safe, # Converte esta coluna para DICTs
    'patentes_similares': parse_list_safe     # Converte esta coluna para LISTAs
}

# Dicionário de 'dtypes' (Tipos de Dados)
# Para forçar o pandas a usar os tipos corretos e mais eficientes
dtype_dict = {
    'lens_id': 'string',  # Mais eficiente que 'object'
    'year': 'Int64',      # 'Int64' (com 'I' maiúsculo) permite valores nulos (NaN)
    'count_similares': 'Int64',
    'emergente': 'boolean'  # 'boolean' (minúsculo) permite valores nulos (NA)
}


print(f"Carregando o arquivo: {caminho_do_arquivo2}...")

try:
    # Carrega o DataFrame aplicando todas as nossas regras
    df_carregado2 = pd.read_csv(
        caminho_do_arquivo2,
        index_col=0,           # A primeira coluna (sem nome) é o índice
        converters=converters_dict,  # Aplica os conversores de dict/list
        dtype=dtype_dict,            # Força os tipos de dados corretos
        low_memory=False       # Recomendado para arquivos complexos
    )

    print("\nArquivo carregado com sucesso!")
    
    print("\nVerificando os tipos de dados das colunas restantes:")
    # Verifica os tipos (dtypes)
    print(df_carregado2.dtypes)
    
    print("\nExemplo de dados convertidos (primeira linha):")
    # Mostra um exemplo dos dados convertidos
    print(f"Tipo 'vetor_caracteristicas': {type(df_carregado2.iloc[0]['vetor_caracteristicas'])}")
    print(f"Tipo 'patentes_similares': {type(df_carregado2.iloc[0]['patentes_similares'])}")

    # Mostra o DataFrame carregado
    print("\nCabeçalho do DataFrame Carregado:")
    display(df_carregado2.head())
    
    # A variável 'df_carregado2' está pronta para ser usada!
    # df_completo = df_carregado2.copy()

except FileNotFoundError:
    print(f"ERRO: O arquivo '{caminho_do_arquivo2}' não foi encontrado.")
except Exception as e:
    print(f"Ocorreu um erro inesperado durante o carregamento: {e}")

Carregando o arquivo: ./resultados_analise/5_topics/analise-de-similaridade2_tanimoto_n5_s0.95.csv...

Arquivo carregado com sucesso!

Verificando os tipos de dados das colunas restantes:
lens_id                  string[python]
year                              Int64
Topic_0                          object
Topic_1                          object
Topic_2                          object
Topic_3                          object
Topic_4                          object
vetor_caracteristicas            object
patentes_similares               object
count_similares                   Int64
emergente                       boolean
dtype: object

Exemplo de dados convertidos (primeira linha):
Tipo 'vetor_caracteristicas': <class 'dict'>
Tipo 'patentes_similares': <class 'list'>

Cabeçalho do DataFrame Carregado:


,lens_id,year,Topic_0,Topic_1,Topic_2,Topic_3,Topic_4,vetor_caracteristicas,patentes_similares,count_similares,emergente
0,171-098-512-075-612,1999,"[[""'estrutura',"", 3.898665454471484e-06], [""'d...","[[""'invenção',"", 9.288967703469098e-06], [""'pa...","[[""'superfície',"", 6.86711155140074e-06], [""'p...","[[""'invenção',"", 6.183001005410915e-06], [""'di...","[[""'invenção',"", 0.009545245207846165], [""'est...","{'estrutura': 3.898665454471484e-06, 'disposit...","[116-121-940-543-96X, 197-447-606-499-762, 169...",9,True
1,046-440-553-666-653,1999,"[[""'outro',"", 4.0580184759164695e-06], [""'pate...","[[""'invenção',"", 8.62777051224839e-06], [""'pat...","[[""'patente',"", 4.253686711308546e-06], [""'inv...","[[""'invenção',"", 0.007665595971047878], [""'fol...","[[""'invenção',"", 8.391961273446213e-06], [""'pa...","{'outro': 4.0580184759164695e-06, 'patente': 3...",[],0,False
2,157-865-805-179-279,1999,"[[""'movimento',"", 2.9853720207029255e-06], [""'...","[[""'invenção',"", 5.626234269584529e-06], [""'co...","[[""'superfície',"", 4.159340278420132e-06], [""'...","[[""'invenção',"", 0.0076750656589865685], [""'di...","[[""'invenção',"", 5.4724605433875695e-06], [""'e...","{'movimento': 2.9853720207029255e-06, 'conjunt...",[],0,False
3,104-140-108-095-175,1999,"[[""'porta',"", 0.038128580898046494], [""'movime...","[[""'invenção',"", 6.123864295659587e-06], [""'el...","[[""'lateral',"", 3.751218400793732e-06], [""'ele...","[[""'elemento',"", 4.126146905036876e-06], [""'in...","[[""'construção',"", 7.65810636949027e-06], [""'e...","{'porta': 0.038128580898046494, 'movimento': 0...",[],0,False
4,023-683-667-869-67X,1999,"[[""'água',"", 2.3737948140478693e-06], [""'conju...","[[""'invenção',"", 4.9905688683793414e-06], [""'m...","[[""'água',"", 4.833969342143973e-06], [""'formar...","[[""'invenção',"", 0.007677071262151003], [""'fol...","[[""'invenção',"", 4.854169674217701e-06], [""'pa...","{'água': 2.3737948140478693e-06, 'conjunto': 2...",[],0,False


### Análise junto da classificação do ipcr

In [31]:
import pandas as pd

filtered_ipcr = pd.read_csv('./datasets/database-filtrado.csv.zip', compression='zip')

In [32]:
import pandas as pd
import ast

df_main = df_carregado.copy()
df_lookup = filtered_ipcr[['lens_id', 'new_ipcr']].copy()

df_lookup['lens_id'] = df_lookup['lens_id'].astype(str).str.strip()

# Passo 1: Criar o Set de consulta rápida
patentes_com_new_ipcr = set(df_lookup[df_lookup['new_ipcr'] == True]['lens_id'])

def safe_convert_list_to_strings(input_data):
    """
    Converte com segurança um item (que pode ser uma lista,
    uma string de lista, ou NaN) em uma lista de strings limpas.
    """
    lista_bruta = [] # Começa com uma lista vazia
    
    try:
        if isinstance(input_data, str):
            # 1. Se for uma string, usa ast.literal_eval para converter
            lista_bruta = ast.literal_eval(input_data)
        elif isinstance(input_data, list):
            # 2. Se já for uma lista, apenas a usa
            lista_bruta = input_data
        # 3. Se for np.nan, None, ou outro tipo, lista_bruta continua []

        # Agora, processa a lista_bruta (se ela for uma lista)
        if isinstance(lista_bruta, list):
            # Converte cada item para string, limpa e remove Nulos
            return [str(item).strip() for item in lista_bruta if pd.notna(item)]
        else:
            # Se ast.literal_eval retornou algo que não é lista (ex: "None")
            return []
            
    except (ValueError, SyntaxError, TypeError):
        # Se ast.literal_eval falhar (ex: string vazia "")
        return []

# Passo 3: Criar uma cópia do DataFrame original
df_resultado = df_main.copy()

# Passo 4: Inicializar a nova coluna no df_resultado
df_resultado['emergent_new_ipcr'] = False

# Passo 5: Identificar o índice das patentes emergentes
idx_emergente = df_resultado[df_resultado['emergente']].index

# Passo 6: Converter a string/lista (usando a NOVA função)
s_similares_lista = df_resultado.loc[idx_emergente, 'patentes_similares'].apply(safe_convert_list_to_strings)

# Passo 7: "Explodir" as listas E REMOVER VAZIOS (NaN)
# .dropna() remove as linhas que vieram de listas vazias (ex: '[]' ou '[None]')
s_explodida = s_similares_lista.explode().dropna()

# Passo 8: Verificar (de forma vetorizada) quais patentes estão no set
s_matches = s_explodida.isin(patentes_com_new_ipcr)

# --- Criação do DataFrame de Comparações ---
df_comparacoes = pd.DataFrame({
    'patente_similar_verificada': s_explodida,
    'teve_new_ipcr': s_matches
})
df_comparacoes['patente_original_lens_id'] = df_comparacoes.index.map(df_resultado['lens_id'])
df_comparacoes = df_comparacoes[['patente_original_lens_id', 'patente_similar_verificada', 'teve_new_ipcr']]
df_comparacoes = df_comparacoes.reset_index(drop=True)

# Passo 9: Agrupar pelo índice original e verificar se ALGUMA deu True
s_resultado_final = s_matches.groupby(level=0).any()

# Passo 10: Atualizar o df_resultado na coluna e linhas corretas
df_resultado.loc[s_resultado_final.index, 'emergent_new_ipcr'] = s_resultado_final

print(f"\nTotal de 'emergent_new_ipcr' = True: {df_resultado['emergent_new_ipcr'].sum()}")


Total de 'emergent_new_ipcr' = True: 2


In [33]:
import pandas as pd
import ast

# (Assumindo que df_carregado2 e filtered_ipcr já existem)
df_main = df_carregado2.copy()
df_lookup = filtered_ipcr[['lens_id', 'new_ipcr']].copy()

df_lookup['lens_id'] = df_lookup['lens_id'].astype(str).str.strip()

# Passo 1: Criar o Set de consulta rápida
patentes_com_new_ipcr = set(df_lookup[df_lookup['new_ipcr'] == True]['lens_id'])

def safe_convert_list_to_strings(input_data):
    """
    Converte com segurança um item (que pode ser uma lista,
    uma string de lista, ou NaN) em uma lista de strings limpas.
    """
    lista_bruta = [] # Começa com uma lista vazia
    
    try:
        if isinstance(input_data, str):
            # 1. Se for uma string, usa ast.literal_eval para converter
            lista_bruta = ast.literal_eval(input_data)
        elif isinstance(input_data, list):
            # 2. Se já for uma lista, apenas a usa
            lista_bruta = input_data
        # 3. Se for np.nan, None, ou outro tipo, lista_bruta continua []

        # Agora, processa a lista_bruta (se ela for uma lista)
        if isinstance(lista_bruta, list):
            # Converte cada item para string, limpa e remove Nulos
            return [str(item).strip() for item in lista_bruta if pd.notna(item)]
        else:
            # Se ast.literal_eval retornou algo que não é lista (ex: "None")
            return []
            
    except (ValueError, SyntaxError, TypeError):
        # Se ast.literal_eval falhar (ex: string vazia "")
        return []

# Passo 3: Criar uma cópia do DataFrame original
df_resultado2 = df_main.copy()

# Passo 4: Inicializar a nova coluna no df_resultado
df_resultado2['emergent_new_ipcr'] = False

# Passo 5: Identificar o índice das patentes emergentes
idx_emergente = df_resultado2[df_resultado2['emergente']].index

# Passo 6: Converter a string/lista (usando a NOVA função)
s_similares_lista = df_resultado2.loc[idx_emergente, 'patentes_similares'].apply(safe_convert_list_to_strings)

# Passo 7: "Explodir" as listas E REMOVER VAZIOS (NaN)
# .dropna() remove as linhas que vieram de listas vazias (ex: '[]' ou '[None]')
s_explodida = s_similares_lista.explode().dropna()

# Passo 8: Verificar (de forma vetorizada) quais patentes estão no set
s_matches = s_explodida.isin(patentes_com_new_ipcr)

# --- Criação do DataFrame de Comparações ---
df_comparacoes = pd.DataFrame({
    'patente_similar_verificada': s_explodida,
    'teve_new_ipcr': s_matches
})
df_comparacoes['patente_original_lens_id'] = df_comparacoes.index.map(df_resultado2['lens_id'])
df_comparacoes = df_comparacoes[['patente_original_lens_id', 'patente_similar_verificada', 'teve_new_ipcr']]
df_comparacoes = df_comparacoes.reset_index(drop=True)

# Passo 9: Agrupar pelo índice original e verificar se ALGUMA deu True
s_resultado_final = s_matches.groupby(level=0).any()

# Passo 10: Atualizar o df_resultado na coluna e linhas corretas
df_resultado2.loc[s_resultado_final.index, 'emergent_new_ipcr'] = s_resultado_final

print(f"\nTotal de 'emergent_new_ipcr' = True: {df_resultado2['emergent_new_ipcr'].sum()}")



Total de 'emergent_new_ipcr' = True: 143


### SVM e XGBoost

In [34]:
colunas_desejadas_ipcr = ['lens_id', 'family_count',
       'claims_count', 'ipcr_publication_years', 'new_ipcr']  

colunas_existentes = [col for col in colunas_desejadas_ipcr if col in filtered_ipcr.columns]
print(f"Colunas disponíveis para merge: {colunas_existentes}")

df_train = pd.merge(
    df_novo, 
    filtered_ipcr[colunas_existentes], 
    on='lens_id',
    how='left'
)

df_train.columns

Colunas disponíveis para merge: ['lens_id', 'family_count', 'claims_count', 'ipcr_publication_years', 'new_ipcr']


Index(['lens_id', 'jurisdiction', 'doc_number', 'kind', 'date_published',
       'doc_key', 'docdb_id', 'lang', 'biblio', 'families', 'legal_status',
       'publication_type', 'abstract', 'abstract_text', 'ipcr_triples',
       'cpc_triples', 'all_triples', 'inventors_detailed', 'inventor_names',
       'invention_title_text', 'patent_count', 'patent_lens_ids',
       'patent_status', 'application_expiry_date', 'picked', 'title_abstract',
       'tokens', 'year', 'family_count', 'claims_count',
       'ipcr_publication_years', 'new_ipcr'],
      dtype='object')

In [38]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.preprocessing import StandardScaler, OneHotEncoder, LabelEncoder
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.svm import SVC
from xgboost import XGBClassifier
from sklearn.metrics import classification_report, accuracy_score
from imblearn.pipeline import Pipeline as ImbPipeline
from imblearn.over_sampling import SMOTE


# --- 1. CONFIGURAÇÃO ---
SMOTE_apply = False
OPTIMIZE_SVM = False
USE_GRID_SEARCH = False 
USE_GRID_SEARCH_PARAMS = True

# df_data = pd.read_csv('.csv')
# df_labels = pd.read_csv('.csv')

df_data = df_train.copy()
df_labels = df_resultado2[['lens_id', 'emergent_new_ipcr']].copy()

print(f"--- Dados Iniciais (Patentes): {df_data.shape} ---")
print(f"\n--- Rótulos Iniciais (Labels): {df_labels.shape} ---")


# --- 2. Preparação e Limpeza dos Dados ---
df_merged = pd.merge(df_data, df_labels, on='lens_id')
print(f"\n--- Dados Combinados: {df_merged.shape} ---")

text_features = ['title_abstract', 'inventor_names']
numeric_features = ['year', 'patent_count', 'family_count', 'claims_count']
categorical_features = ['jurisdiction', 'kind', 'lang', 'publication_type', 'patent_status']

df_merged[text_features] = df_merged[text_features].fillna('')
df_merged[numeric_features] = df_merged[numeric_features].fillna(0) 
df_merged[categorical_features] = df_merged[categorical_features].fillna('Missing') 

X = df_merged[text_features + numeric_features + categorical_features]
y_raw = df_merged['emergent_new_ipcr']

le = LabelEncoder()
y = le.fit_transform(y_raw)


# --- 3. Engenharia de Features (Pré-processamento) ---
# (Usando a nossa melhor versão com ngram_range=(1, 2))
text_transformer = TfidfVectorizer(max_features=5000, 
                                   ngram_range=(1, 2) 
                                  )

numeric_transformer = StandardScaler()
categorical_transformer = OneHotEncoder(handle_unknown='ignore') 

preprocessor = ColumnTransformer(
    transformers=[
        ('text', text_transformer, text_features[0]), 
        ('num', numeric_transformer, numeric_features),
        ('cat', categorical_transformer, categorical_features)
    ])


# --- 4. Divisão dos Dados (Treino e Teste) ---
X_train, X_test, y_train, y_test = train_test_split(X, y, 
                                                     test_size=0.2, 
                                                     random_state=42, 
                                                     stratify=y) 

count_neg = np.sum(y_train == 0)
count_pos = np.sum(y_train == 1)
scale_weight = count_neg / count_pos

print(f"\nDivisão dos dados de TREINO:")
print(f"  Classe Negativa (False): {count_neg}")
print(f"  Classe Positiva (True):  {count_pos}")
print(f"  Proporção (scale_pos_weight): {scale_weight:.2f}")


# --- 5. Modelo 1: SVM ---

if SMOTE_apply:
    print("\n--- Treinando Modelo 1: SVM (com SMOTE) ---")
    svm_pipeline_smote = ImbPipeline(steps=[
        ('preprocessor', preprocessor),
        ('smote', SMOTE(random_state=42)),
        ('classifier', SVC(kernel='linear', random_state=42)) 
    ])
    svm_pipeline_smote.fit(X_train, y_train)
    y_pred_svm = svm_pipeline_smote.predict(X_test)
else:
    # --- Ramo NÃO-SMOTE (Balanceado) ---
    if OPTIMIZE_SVM:
        print("\n--- Treinando Modelo 1: SVM (Otimizado com GridSearchCV) ---")
        
        svm_pipeline_for_grid = Pipeline(steps=[
            ('preprocessor', preprocessor),
            ('classifier', SVC(random_state=42,
                               class_weight='balanced'
                              ))
        ])

        # Grade de parâmetros para o SVM
        svm_param_grid = {
            'classifier__kernel': ['linear', 'rbf'], # Testar kernel linear e rbf
            'classifier__C': [0.1, 1, 10]           # Testar diferentes custos
        }

        svm_grid_search = GridSearchCV(estimator=svm_pipeline_for_grid, 
                                       param_grid=svm_param_grid, 
                                       cv=3, 
                                       scoring='f1_macro', # Otimizar pela mesma métrica
                                       n_jobs=-1,
                                       verbose=2)

        print("Iniciando o GridSearch para o SVM...")
        svm_grid_search.fit(X_train, y_train)
        
        print("Avaliando o melhor modelo SVM...")
        y_pred_svm = svm_grid_search.predict(X_test) # Salva a predição
    
    else:
        if USE_GRID_SEARCH_PARAMS:
            print("\n--- Treinando Modelo 1: SVM (Balanceado - Único com Parâmetros de GridSearch) ---")
            svm_pipeline = Pipeline(steps=[
                ('preprocessor', preprocessor),
                ('classifier', SVC(kernel='rbf', 
                                    random_state=42,
                                    class_weight='balanced',
                                    C=1
                                    ))
            ])
            svm_pipeline.fit(X_train, y_train)
            print("Avaliando SVM no conjunto de teste...")
            y_pred_svm = svm_pipeline.predict(X_test)
        else:
            print("\n--- Treinando Modelo 1: SVM (Balanceado - Único) ---")
            svm_pipeline = Pipeline(steps=[
                ('preprocessor', preprocessor),
                ('classifier', SVC(kernel='linear', 
                                    random_state=42,
                                    class_weight='balanced' 
                                    )) 
            ])
            svm_pipeline.fit(X_train, y_train)
            print("Avaliando SVM no conjunto de teste...")
            y_pred_svm = svm_pipeline.predict(X_test)


# --- 6. Modelo 2: XGBoost ---
if SMOTE_apply:
    print("\n--- Treinando Modelo 2: XGBoost (com SMOTE) ---")
    xgb_pipeline_smote = ImbPipeline(steps=[
        ('preprocessor', preprocessor),
        ('smote', SMOTE(random_state=42)),
        ('classifier', XGBClassifier(use_label_encoder=False, 
                                       eval_metric='logloss', 
                                       random_state=42,
                                       n_estimators=100
                                      ))
    ])
    xgb_pipeline_smote.fit(X_train, y_train)
    y_pred_xgb = xgb_pipeline_smote.predict(X_test)
else:
    if USE_GRID_SEARCH:
        print("\n--- Treinando Modelo 2: XGBoost (Otimizado com GridSearchCV) ---")
        
        xgb_pipeline_for_grid = Pipeline(steps=[
            ('preprocessor', preprocessor),
            ('classifier', XGBClassifier(use_label_encoder=False, 
                                         eval_metric='logloss', 
                                         random_state=42,
                                         scale_pos_weight=scale_weight
                                        ))
        ])

        param_grid = {
            'classifier__n_estimators': [100, 250, 500],
            'classifier__max_depth': [3, 5, 7],
            'classifier__learning_rate': [0.1, 0.05]
        }

        grid_search = GridSearchCV(estimator=xgb_pipeline_for_grid, 
                                   param_grid=param_grid, 
                                   cv=3, 
                                   scoring='f1_macro', 
                                   n_jobs=-1,
                                   verbose=2)

        print("Iniciando o GridSearch para o XGBoost... (com ngrams 1,2)")
        grid_search.fit(X_train, y_train)
        
        print("Avaliando o melhor modelo XGBoost...")
        y_pred_xgb = grid_search.predict(X_test)

    else:
        if USE_GRID_SEARCH_PARAMS:
            print("\n--- Treinando Modelo 2: XGBoost (Balanceado - Único com Parâmetros de GridSearch) ---")
            xgb_pipeline = Pipeline(steps=[
                ('preprocessor', preprocessor),
                ('classifier', XGBClassifier(use_label_encoder=False, 
                                            eval_metric='logloss', 
                                            random_state=42,
                                            n_estimators=500,
                                            scale_pos_weight=scale_weight,
                                            max_depth=3,
                                            learning_rate=0.1
                                            ))
            ])
            xgb_pipeline.fit(X_train, y_train)
            print("Avaliando XGBoost no conjunto de teste...")
            y_pred_xgb = xgb_pipeline.predict(X_test)
        else:
            print("\n--- Treinando Modelo 2: XGBoost (Balanceado - Único) ---")
            xgb_pipeline = Pipeline(steps=[
                ('preprocessor', preprocessor),
                ('classifier', XGBClassifier(use_label_encoder=False, 
                                            eval_metric='logloss', 
                                            random_state=42,
                                            n_estimators=100,
                                            scale_pos_weight=scale_weight
                                            ))
            ])
            xgb_pipeline.fit(X_train, y_train)
            print("Avaliando XGBoost no conjunto de teste...")
            y_pred_xgb = xgb_pipeline.predict(X_test)


# --- 7. Avaliação dos Modelos ---

target_names = le.classes_.astype(str) # ['False', 'True']

if SMOTE_apply:
    print("\n\n--- RESULTADOS DA AVALIAÇÃO (Modelos com SMOTE) ---")
    print("\n--- Modelo SVM (SMOTE) ---")
    print(f"Acurácia: {accuracy_score(y_test, y_pred_svm):.4f}")
    print("Relatório de Classificação:")
    print(classification_report(y_test, y_pred_svm, target_names=target_names))

    print("\n--- Modelo XGBoost (SMOTE) ---")
    print(f"Acurácia: {accuracy_score(y_test, y_pred_xgb):.4f}")
    print("Relatório de Classificação:")
    print(classification_report(y_test, y_pred_xgb, target_names=target_names))
else:
    print("\n\n--- RESULTADOS DA AVALIAÇÃO (Modelos Balanceados) ---")
    
    if OPTIMIZE_SVM:
        print("\n--- Modelo SVM (Otimizado via GridSearch) ---")
        print(f"Melhores Parâmetros Encontrados: {svm_grid_search.best_params_}")
    else:
        print("\n--- Modelo SVM (Balanceado - Único) ---")
    
    print(f"Acurácia: {accuracy_score(y_test, y_pred_svm):.4f}")
    print("Relatório de Classificação:")
    print(classification_report(y_test, y_pred_svm, target_names=target_names))
    
    
    if USE_GRID_SEARCH:
        print("\n--- Modelo XGBoost (Otimizado via GridSearch) ---")
        print(f"Melhores Parâmetros Encontrados: {grid_search.best_params_}")
    else:
        print("\n--- Modelo XGBoost (Balanceado - Único) ---")
    
    print(f"Acurácia: {accuracy_score(y_test, y_pred_xgb):.4f}")
    print("Relatório de Classificação:")
    print(classification_report(y_test, y_pred_xgb, target_names=target_names))

--- Dados Iniciais (Patentes): (25240, 32) ---

--- Rótulos Iniciais (Labels): (25240, 2) ---

--- Dados Combinados: (25240, 33) ---

Divisão dos dados de TREINO:
  Classe Negativa (False): 20078
  Classe Positiva (True):  114
  Proporção (scale_pos_weight): 176.12

--- Treinando Modelo 1: SVM (Balanceado - Único com Parâmetros de GridSearch) ---


KeyboardInterrupt: 